# 122 · Experimental Setup — Modelo 120

Banco de pruebas para documentar y medir el protocolo experimental del modelo `120_chess_siamese_instance_memory.ipynb`.

Este notebook **no reentrena** el modelo. Carga el checkpoint del `120`, usa el embedding aprendido y regenera bancos de memoria para estudiar:

- **A1.** Cómo se eligió el pool base de `N=10`: aleatorio, pero cubriendo el rango de ELO.
- **A2.** Repetibilidad del muestreo con `5` seeds distintas: se crean `5` pools base distintos de `10` jugadores, se regeneran memory banks y se reporta MAE medio + desviación.
- **A3.** Cuántas partidas del jugador-query hacen falta: `nq ∈ {1, 3, 5, 10, 20}` y curva MAE vs `nq`.

Nota metodológica: el ELO **no entra en el entrenamiento del embedding**. En esta batería se usa para seleccionar pools estratificados, para asignar ELO conocido a los jugadores de referencia al construir la estimación final, y para ajustar una calibración lineal post-hoc.


## 0) Configuración

Ajusta aquí el tamaño de la batería si necesitas una ejecución más rápida o más rigurosa. Los valores por defecto están pensados para que el protocolo sea representativo sin llegar al coste del notebook masivo `122_chess_siamese_elo_open_scale.ipynb`.


In [ ]:
import gc
import hashlib
import io
import json
import math
import random
import re
import sqlite3
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import chess
import chess.pgn
import chess.svg
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
import zstandard as zstd

SEED = 314159
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.style.use("seaborn-v0_8")
sns.set_context("talk")


@dataclass
class ExperimentalConfig:
    event_name: str = "compressed_120"
    eval_name: str = "experimental_setup_122"

    sequence_len: int = 15
    fullmove_start: int = 15
    fullmove_end: int = 60
    image_size: Tuple[int, int] = (192, 192)
    embedding_dim: int = 256
    infer_batch_size: int = 8

    base_pool_size: int = 10
    memory_games_per_player: int = 20
    query_players_per_seed: int = 80
    query_games_per_player: int = 20
    n_query_grid: Tuple[int, ...] = (1, 3, 5, 10, 20)
    seeds: Tuple[int, ...] = (101, 202, 303, 404, 505)

    min_games_in_db: int = 80
    catalog_target_with_elo: int = 2500
    catalog_query_limit: int = 40000
    elo_min: int = 700
    elo_max: int = 2800

    exemplar_knn_k: int = 25
    player_memory_top_k: int = 5
    player_regression_k: int = 5
    centroid_regression_k: int = 5
    weight_epsilon: float = 1e-6
    calibration_fraction: float = 0.35

    refresh_catalog: bool = False
    resume_seed_predictions: bool = True
    max_games_to_scan_per_player: Optional[int] = None


CONFIG = ExperimentalConfig()
PLY_LABELS = [f"ply{i:02d}" for i in range(1, CONFIG.sequence_len + 1)]

print(f"TensorFlow {tf.__version__}")
print(CONFIG)


## A1) Pool base `N=10` y uso del ELO

El notebook `120` selecciona `10` jugadores con un muestreo aleatorio condicionado por ELO: primero obtiene candidatos con suficientes partidas, extrae un ELO aproximado desde los headers PGN y después escoge jugadores repartidos por bins de ELO.

La idea experimental es evitar que el pool quede concentrado en un único tramo de fuerza. El ELO se usa para construir un conjunto base variado y para analizar el nivel estimado, pero **no se entrega al modelo durante el entrenamiento**: las entradas del embedding son las imágenes de tablero y heatmap.


In [ ]:
def detect_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "labs").exists():
            return candidate
    raise RuntimeError("No se encontró la raíz del repo (carpeta labs/).")


def first_existing(candidates: List[Path], description: str) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    joined = "\n".join(str(c) for c in candidates)
    raise RuntimeError(f"No se encontró {description}. Candidatos:\n{joined}")


def discover_event_players(*dir_pairs: Tuple[Path, Path]) -> List[str]:
    player_sets = []
    for board_dir, heat_dir in dir_pairs:
        if not board_dir.exists() or not heat_dir.exists():
            continue
        players = {
            d.name
            for d in board_dir.iterdir()
            if d.is_dir() and (heat_dir / d.name).is_dir()
        }
        if players:
            player_sets.append(players)

    if not player_sets:
        raise RuntimeError("No se pudieron inferir jugadores del evento 120.")

    common_players = set.intersection(*player_sets) if len(player_sets) > 1 else player_sets[0]
    out = sorted(common_players)
    if not out:
        raise RuntimeError("La intersección TRAIN/HOLDOUT del evento está vacía.")
    return out


REPO_ROOT = detect_repo_root()
LABS_DIR = REPO_ROOT / "labs"

EVENT_DIR = first_existing(
    [
        LABS_DIR / "notebooks" / "output" / "events" / CONFIG.event_name,
        Path("/workspace/code/labs/notebooks/output/events") / CONFIG.event_name,
        Path("/mnt/d/CODE/jupyter/labs/notebooks/output/events") / CONFIG.event_name,
        Path("D:/CODE/jupyter/labs/notebooks/output/events") / CONFIG.event_name,
    ],
    "directorio del evento 120",
)

TRAIN_BOARD_DIR = EVENT_DIR / "board_images"
TRAIN_HEAT_DIR = EVENT_DIR / "heatmap_images"
HOLDOUT_BOARD_DIR = EVENT_DIR / "holdout_board_images"
HOLDOUT_HEAT_DIR = EVENT_DIR / "holdout_heatmap_images"

TRAINING_STATE_DIR = EVENT_DIR / "training_state"
CHECKPOINTS_DIR = TRAINING_STATE_DIR / "checkpoints"
BEST_WEIGHTS_PATH = CHECKPOINTS_DIR / "metric_trainer_best.weights.h5"
LAST_WEIGHTS_PATH = CHECKPOINTS_DIR / "metric_trainer_last.weights.h5"

INDEX_DB = first_existing(
    [
        Path("/pgn_data/index.db"),
        Path("/workspace/pgn_output/index.db"),
        Path("/mnt/d/pgn_output/index.db"),
        Path("D:/pgn_output/index.db"),
    ],
    "index.db de pgn_output",
)
ZST_PLAYERS_DIR = first_existing(
    [
        Path("/pgn_data/players"),
        Path("/workspace/pgn_output/players"),
        Path("/mnt/d/pgn_output/players"),
        Path("D:/pgn_output/players"),
    ],
    "directorio players/ de pgn_output",
)

EVAL_DIR = LABS_DIR / "notebooks" / "output" / "events" / CONFIG.eval_name
SEED_DIR = EVAL_DIR / "seed_predictions"
EMBED_CACHE_DIR = EVAL_DIR / "embedding_cache"
for path in [EVAL_DIR, SEED_DIR, EMBED_CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

CATALOG_CSV_PATH = EVAL_DIR / "candidate_catalog.csv"
BASE_POOLS_CSV_PATH = EVAL_DIR / "base_pools_by_seed.csv"
QUERY_POOLS_CSV_PATH = EVAL_DIR / "query_pools_by_seed.csv"
A2_METRICS_CSV_PATH = EVAL_DIR / "a2_seed_metrics.csv"
A3_METRICS_CSV_PATH = EVAL_DIR / "a3_nq_metrics.csv"
RUN_META_PATH = EVAL_DIR / "run_meta.json"

TRAINED_PLAYERS = discover_event_players(
    (TRAIN_BOARD_DIR, TRAIN_HEAT_DIR),
    (HOLDOUT_BOARD_DIR, HOLDOUT_HEAT_DIR),
)

print(f"Repo: {REPO_ROOT}")
print(f"Evento 120: {EVENT_DIR}")
print(f"Checkpoint dir: {CHECKPOINTS_DIR}")
print(f"index.db: {INDEX_DB}")
print(f"players/: {ZST_PLAYERS_DIR}")
print(f"Salida eval: {EVAL_DIR}")
print(f"Jugadores del entrenamiento 120 ({len(TRAINED_PLAYERS)}): {TRAINED_PLAYERS}")


## 1) Carga del modelo 120

Se reconstruye la arquitectura del `120` y se cargan sus pesos. Para evitar descargas, `ResNet50` se instancia con `weights=None`: el checkpoint local repone los pesos aprendidos/cargados por el entrenamiento original.


In [ ]:
def build_embedding_model(
    seq_len: int, image_size: Tuple[int, int], embedding_dim: int
) -> tf.keras.Model:
    h, w = image_size

    board_input = tf.keras.layers.Input(
        shape=(seq_len, h, w, 3), name="board_sequence"
    )
    heat_input = tf.keras.layers.Input(
        shape=(seq_len, h, w, 1), name="heat_sequence"
    )

    base_cnn = tf.keras.applications.ResNet50(
        include_top=False,
        weights=None,
        pooling="avg",
        input_shape=(h, w, 3),
    )
    base_cnn.trainable = False

    board_pre = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Lambda(
            lambda x: tf.keras.applications.resnet50.preprocess_input(x * 255.0)
        ),
        name="board_preprocess",
    )(board_input)
    board_features = tf.keras.layers.TimeDistributed(
        base_cnn, name="board_backbone"
    )(board_pre)
    board_features = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(512, activation="gelu"), name="board_dense"
    )(board_features)

    heat_encoder = tf.keras.Sequential(
        [
            tf.keras.layers.Conv2D(32, 3, padding="same", activation="relu"),
            tf.keras.layers.MaxPooling2D(2),
            tf.keras.layers.Conv2D(64, 3, padding="same", activation="relu"),
            tf.keras.layers.MaxPooling2D(2),
            tf.keras.layers.Conv2D(128, 3, padding="same", activation="relu"),
            tf.keras.layers.GlobalAveragePooling2D(),
            tf.keras.layers.Dense(256, activation="gelu"),
        ],
        name="heat_encoder",
    )
    heat_features = tf.keras.layers.TimeDistributed(
        heat_encoder, name="heat_backbone"
    )(heat_input)

    fused = tf.keras.layers.Concatenate(name="fuse_modalities")(
        [board_features, heat_features]
    )
    fused = tf.keras.layers.LayerNormalization(name="fuse_norm")(fused)
    fused = tf.keras.layers.TimeDistributed(
        tf.keras.layers.Dense(512, activation="gelu"), name="fuse_dense"
    )(fused)

    temporal = tf.keras.layers.Bidirectional(
        tf.keras.layers.GRU(
            256,
            return_sequences=True,
            dropout=0.20,
            reset_after=False,
        ),
        name="temporal_gru",
    )(fused)
    pooled = tf.keras.layers.GlobalAveragePooling1D(name="temporal_pool")(temporal)

    x = tf.keras.layers.Dense(512, activation="gelu")(pooled)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(embedding_dim, activation=None, dtype="float32")(x)
    embedding = tf.keras.layers.Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1), name="embedding_l2"
    )(x)

    return tf.keras.Model(
        inputs=[board_input, heat_input],
        outputs=embedding,
        name="stylometry_embedding_104",
    )


class MetricTrainer(tf.keras.Model):
    def __init__(
        self,
        embedding_model: tf.keras.Model,
        num_classes: int,
        margin: float = 0.06,
        ce_weight: float = 0.20,
    ):
        super().__init__()
        self.embedding_model = embedding_model
        self.classifier = tf.keras.layers.Dense(
            num_classes,
            use_bias=False,
            dtype="float32",
            name="classifier",
        )
        self.margin = float(margin)
        self.ce_weight = float(ce_weight)

    def call(self, inputs, training=False):
        emb = self.embedding_model(inputs, training=training)
        logits = self.classifier(emb)
        return emb, logits


def pick_checkpoint_path() -> Path:
    candidates = [
        BEST_WEIGHTS_PATH,
        LAST_WEIGHTS_PATH,
        *sorted(CHECKPOINTS_DIR.glob("*.weights.h5")),
        *sorted(TRAINING_STATE_DIR.glob("*.weights.h5")),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise RuntimeError("No se encontró checkpoint de pesos (.weights.h5).")


CHECKPOINT_PATH = pick_checkpoint_path()

embedding_model = build_embedding_model(
    seq_len=CONFIG.sequence_len,
    image_size=CONFIG.image_size,
    embedding_dim=CONFIG.embedding_dim,
)
trainer = MetricTrainer(
    embedding_model=embedding_model,
    num_classes=len(TRAINED_PLAYERS),
)

dummy_inputs = {
    "board_sequence": np.zeros(
        (1, CONFIG.sequence_len, CONFIG.image_size[0], CONFIG.image_size[1], 3),
        dtype=np.float32,
    ),
    "heat_sequence": np.zeros(
        (1, CONFIG.sequence_len, CONFIG.image_size[0], CONFIG.image_size[1], 1),
        dtype=np.float32,
    ),
}
_ = trainer(dummy_inputs, training=False)
trainer.load_weights(str(CHECKPOINT_PATH))
embedding_model = trainer.embedding_model

print(f"Pesos cargados desde: {CHECKPOINT_PATH}")


## 2) Preprocesado PGN → tensores del modelo

Estas funciones reproducen el criterio del `120`: tablero RGB desde la perspectiva del jugador y heatmap de origen/destino ponderado por tiempo de decisión. No se reutilizan imágenes preexistentes para los nuevos bancos; se regeneran tensores desde PGN y se pasa cada partida por el embedding del modelo 120.


In [ ]:
def board_to_rgb_array(board: chess.Board, size: int = 400) -> np.ndarray:
    try:
        import cairosvg
    except ImportError as exc:
        raise ImportError("Falta cairosvg. Instala: pip install cairosvg") from exc

    svg = chess.svg.board(board=board, size=size, coordinates=False)
    png_bytes = cairosvg.svg2png(bytestring=svg.encode("utf-8"))
    png_array = np.frombuffer(png_bytes, dtype=np.uint8)
    bgr = cv2.imdecode(png_array, cv2.IMREAD_COLOR)
    if bgr is None:
        raise RuntimeError("No se pudo renderizar el tablero a imagen.")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def parse_clock_time(clk_string: str) -> Optional[float]:
    if not clk_string:
        return None
    hms = re.match(r"(\d+):(\d+):(\d+)", clk_string)
    if hms:
        h, m, s = map(int, hms.groups())
        return float(h * 3600 + m * 60 + s)
    ms = re.match(r"(\d+):(\d+)", clk_string)
    if ms:
        m, s = map(int, ms.groups())
        return float(m * 60 + s)
    return None


def extract_time_control(game: chess.pgn.Game) -> Optional[Tuple[int, int]]:
    tc_header = game.headers.get("TimeControl", "")
    match = re.match(r"(\d+)\+(\d+)", tc_header)
    if not match:
        return None
    base, inc = map(int, match.groups())
    return base, inc


def extract_decision_seconds_by_halfmove(game: chess.pgn.Game) -> Dict[int, Dict[str, float]]:
    tc = extract_time_control(game)
    if tc is None:
        return {}

    initial_seconds, increment = tc
    prev_clock = {"white": None, "black": None}
    board = game.board()
    move_idx = 0
    out: Dict[int, Dict[str, float]] = {}

    for node in game.mainline():
        move_idx += 1
        color = "white" if board.turn == chess.WHITE else "black"
        board.push(node.move)

        clk_match = re.search(r"\[%clk\s+(\d+:\d+:\d+|\d+:\d+)\]", node.comment or "")
        if not clk_match:
            continue

        clk = parse_clock_time(clk_match.group(1))
        if clk is None:
            continue

        if prev_clock[color] is not None:
            elapsed = max(prev_clock[color] - clk - increment, 0.0)
            out[move_idx] = {
                "color": color,
                "seconds": float(elapsed),
                "relative": float(elapsed / max(initial_seconds, 1)),
            }

        prev_clock[color] = clk

    return out


def _norm_player(name: str) -> str:
    return name.strip().lower().replace(" ", "_")


def resolve_player_color(game: chess.pgn.Game, player_name: str) -> Optional[chess.Color]:
    token = _norm_player(player_name)
    white = _norm_player(game.headers.get("White", ""))
    black = _norm_player(game.headers.get("Black", ""))
    if token == white:
        return chess.WHITE
    if token == black:
        return chess.BLACK
    return None


def select_player_halfmoves(
    total_halfmoves: int,
    player_color: chess.Color,
    fullmove_start: int,
    fullmove_end: int,
) -> List[int]:
    selected: List[int] = []
    for hm in range(1, total_halfmoves + 1):
        fullmove = (hm + 1) // 2
        if not (fullmove_start <= fullmove <= fullmove_end):
            continue
        color = chess.WHITE if hm % 2 == 1 else chess.BLACK
        if color == player_color:
            selected.append(hm)
    return selected


def normalize_board_for_player(board: chess.Board, player_color: chess.Color) -> chess.Board:
    return board.copy() if player_color == chess.WHITE else board.mirror()


def build_player_move_heatmap(
    move: chess.Move,
    decision_seconds: float,
    output_size: int,
    mirror_for_black: bool = False,
) -> np.ndarray:
    grid = np.zeros((8, 8), dtype=np.float32)

    from_sq = move.from_square
    to_sq = move.to_square
    if mirror_for_black:
        from_sq = chess.square_mirror(from_sq)
        to_sq = chess.square_mirror(to_sq)

    fr, fc = 7 - (from_sq // 8), from_sq % 8
    tr, tc = 7 - (to_sq // 8), to_sq % 8

    norm = float(np.clip(np.log1p(decision_seconds) / np.log1p(60.0), 0.0, 1.0))
    origin_val = 0.25 + 0.75 * norm
    target_val = 0.12 + 0.45 * norm

    grid[fr, fc] = max(grid[fr, fc], origin_val)
    grid[tr, tc] = max(grid[tr, tc], target_val)

    image = (255.0 * grid).astype(np.uint8)
    return cv2.resize(image, (output_size, output_size), interpolation=cv2.INTER_NEAREST)


def build_single_game_tensor(
    game: chess.pgn.Game,
    player_name: str,
    sequence_len: int,
    fullmove_start: int,
    fullmove_end: int,
    image_size: Tuple[int, int],
) -> Tuple[np.ndarray, np.ndarray, str]:
    player_color = resolve_player_color(game, player_name)
    if player_color is None:
        raise RuntimeError(f"La partida no pertenece a {player_name}.")

    moves = list(game.mainline_moves())
    selected_halfmoves = select_player_halfmoves(
        total_halfmoves=len(moves),
        player_color=player_color,
        fullmove_start=fullmove_start,
        fullmove_end=fullmove_end,
    )
    if len(selected_halfmoves) < sequence_len:
        raise RuntimeError("La partida no tiene suficientes jugadas para esta configuración.")

    selected_halfmoves = selected_halfmoves[:sequence_len]
    decision_map = extract_decision_seconds_by_halfmove(game)

    board = game.board()
    board_states: Dict[int, chess.Board] = {}
    for hm, move in enumerate(moves, start=1):
        board.push(move)
        if hm in selected_halfmoves:
            board_states[hm] = board.copy()
        if len(board_states) == len(selected_halfmoves):
            break

    h, w = image_size
    boards = np.zeros((sequence_len, h, w, 3), dtype=np.float32)
    heats = np.zeros((sequence_len, h, w, 1), dtype=np.float32)

    expected_color_name = "white" if player_color == chess.WHITE else "black"
    for idx, hm in enumerate(selected_halfmoves):
        state = normalize_board_for_player(board_states[hm], player_color)
        rgb = board_to_rgb_array(state, size=400)
        rgb = cv2.resize(rgb, (w, h), interpolation=cv2.INTER_AREA)
        boards[idx] = rgb.astype(np.float32) / 255.0

        move = moves[hm - 1]
        info = decision_map.get(hm, {})
        seconds = (
            float(info.get("seconds", 0.0))
            if info.get("color") == expected_color_name
            else 0.0
        )
        heat = build_player_move_heatmap(
            move=move,
            decision_seconds=seconds,
            output_size=h,
            mirror_for_black=(player_color == chess.BLACK),
        )
        heats[idx, :, :, 0] = heat.astype(np.float32) / 255.0

    side = "white" if player_color == chess.WHITE else "black"
    return boards, heats, side


def iter_games_from_zst(zst_path: Path) -> Iterable[Tuple[int, str]]:
    dctx = zstd.ZstdDecompressor()
    game_counter = 0
    with open(zst_path, "rb") as f:
        with dctx.stream_reader(f) as reader:
            text_reader = io.TextIOWrapper(reader, encoding="utf-8", errors="replace")
            current_lines: List[str] = []

            for line in text_reader:
                if line.startswith("[Event ") and current_lines:
                    game_counter += 1
                    yield game_counter, "".join(current_lines)
                    current_lines = [line]
                else:
                    current_lines.append(line)

            if current_lines:
                game_counter += 1
                yield game_counter, "".join(current_lines)


def stable_int_seed(*parts: object) -> int:
    payload = "|".join(str(p) for p in parts)
    digest = hashlib.md5(payload.encode("utf-8")).hexdigest()
    return int(digest[:8], 16)


def sample_random_games_from_zst(
    zst_path: Path,
    player_name: str,
    sample_size: int,
    seed: int,
    max_games_to_scan: Optional[int] = None,
) -> Tuple[List[Tuple[int, chess.pgn.Game]], int, int]:
    rng = random.Random(seed)
    reservoir: List[Tuple[int, chess.pgn.Game]] = []
    useful_seen = 0
    scanned_total = 0

    for game_index, game_text in iter_games_from_zst(zst_path):
        scanned_total += 1
        if max_games_to_scan is not None and scanned_total > max_games_to_scan:
            break

        game = chess.pgn.read_game(io.StringIO(game_text))
        if game is None or resolve_player_color(game, player_name) is None:
            continue

        useful_seen += 1
        item = (game_index, game)
        if len(reservoir) < sample_size:
            reservoir.append(item)
        else:
            draw = rng.randint(1, useful_seen)
            if draw <= sample_size:
                reservoir[draw - 1] = item

    reservoir.sort(key=lambda x: x[0])
    return reservoir, scanned_total, useful_seen


## 3) Catálogo y selección estratificada por ELO

El catálogo es solo una fuente de jugadores candidatos con ELO extraído de PGN. Para cada seed se selecciona un pool base de `N=10` dividiendo el catálogo ordenado por ELO en `10` tramos y tomando un jugador aleatorio de cada tramo. Así el muestreo sigue siendo aleatorio, pero cubre el rango completo.


In [ ]:
def find_zst_path(player_name: str) -> Optional[Path]:
    prefix = player_name[:2].lower()
    candidate = ZST_PLAYERS_DIR / prefix / f"{player_name}.pgn.zst"
    if candidate.exists():
        return candidate

    prefix_dir = ZST_PLAYERS_DIR / prefix
    if not prefix_dir.exists():
        return None

    target = f"{player_name.lower()}.pgn.zst"
    for file_path in prefix_dir.iterdir():
        if file_path.name.lower() == target:
            return file_path
    return None


def extract_elo_from_zst(zst_path: Path, max_bytes: int = 120_000) -> Optional[int]:
    try:
        dctx = zstd.ZstdDecompressor()
        with open(zst_path, "rb") as f:
            reader = dctx.stream_reader(f)
            text = reader.read(max_bytes).decode("utf-8", errors="replace")

        values: List[int] = []
        for line in text.splitlines():
            line = line.strip()
            if line.startswith("[WhiteElo") or line.startswith("[BlackElo"):
                try:
                    elo = int(line.split('"')[1])
                except (IndexError, ValueError):
                    continue
                if 400 <= elo <= 3500:
                    values.append(elo)

        if values:
            return int(np.median(values))
    except Exception:
        return None
    return None


def load_original_120_pool() -> pd.DataFrame:
    rows = []
    for player in TRAINED_PLAYERS:
        zst_path = find_zst_path(player)
        elo = extract_elo_from_zst(zst_path) if zst_path else None
        rows.append(
            {
                "player": player,
                "elo": elo,
                "zst_path": str(zst_path) if zst_path else None,
                "source": "trained_pool_120",
            }
        )
    return pd.DataFrame(rows).sort_values("elo").reset_index(drop=True)


def build_candidate_catalog(refresh: bool = False) -> pd.DataFrame:
    if CATALOG_CSV_PATH.exists() and not refresh:
        catalog_df = pd.read_csv(CATALOG_CSV_PATH)
        if not catalog_df.empty:
            print(f"Usando catálogo existente: {CATALOG_CSV_PATH}")
            return catalog_df

    known_norm = {_norm_player(p) for p in TRAINED_PLAYERS}
    rows = []

    conn = sqlite3.connect(str(INDEX_DB))
    try:
        sql = (
            "SELECT name, total_games FROM players "
            "WHERE total_games >= ? "
            "ORDER BY total_games DESC LIMIT ?"
        )
        candidate_rows = conn.execute(
            sql,
            (int(CONFIG.min_games_in_db), int(CONFIG.catalog_query_limit)),
        ).fetchall()
    finally:
        conn.close()

    print(f"Candidatos SQL: {len(candidate_rows)}")
    seen = set()
    for idx, (player_name, total_games) in enumerate(candidate_rows, start=1):
        player_name = str(player_name)
        player_norm = _norm_player(player_name)
        if player_norm in known_norm or player_norm in seen:
            continue
        seen.add(player_norm)

        zst_path = find_zst_path(player_name)
        if zst_path is None:
            continue

        elo = extract_elo_from_zst(zst_path)
        if elo is None or not (CONFIG.elo_min <= elo <= CONFIG.elo_max):
            continue

        rows.append(
            {
                "player": player_name,
                "total_games": int(total_games),
                "elo": int(elo),
                "zst_path": str(zst_path),
            }
        )
        if len(rows) >= CONFIG.catalog_target_with_elo:
            break

        if idx % 1000 == 0:
            print(f"  revisados={idx} | con_elo={len(rows)}")

    if not rows:
        raise RuntimeError("No se pudo construir catálogo con ELO válido.")

    catalog_df = (
        pd.DataFrame(rows)
        .drop_duplicates(subset=["player"])
        .sort_values(["elo", "total_games", "player"])
        .reset_index(drop=True)
    )
    catalog_df.to_csv(CATALOG_CSV_PATH, index=False)
    print(f"Catálogo guardado en {CATALOG_CSV_PATH} con {len(catalog_df)} jugadores.")
    return catalog_df


def select_elo_covered_pool(
    catalog_df: pd.DataFrame,
    n: int,
    seed: int,
    exclude_players: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    exclude_norm = {_norm_player(p) for p in (exclude_players or [])}
    work = catalog_df.loc[
        ~catalog_df["player"].map(_norm_player).isin(exclude_norm)
    ].copy()
    work = work.sort_values("elo").reset_index(drop=True)
    if len(work) < n:
        raise RuntimeError(f"Catálogo insuficiente tras exclusiones: {len(work)} < {n}")

    rng = random.Random(seed)
    chunks = np.array_split(np.arange(len(work)), n)
    selected_rows = []
    used_idx = set()
    for chunk in chunks:
        if len(chunk) == 0:
            continue
        idx = int(rng.choice(list(chunk)))
        selected_rows.append(work.iloc[idx].to_dict())
        used_idx.add(idx)

    if len(selected_rows) < n:
        remaining = [i for i in range(len(work)) if i not in used_idx]
        rng.shuffle(remaining)
        for idx in remaining[: n - len(selected_rows)]:
            selected_rows.append(work.iloc[idx].to_dict())

    out = pd.DataFrame(selected_rows).sort_values("elo").reset_index(drop=True)
    out["selection_seed"] = int(seed)
    return out


original_pool_df = load_original_120_pool()
catalog_df = build_candidate_catalog(refresh=CONFIG.refresh_catalog)

display(original_pool_df[["player", "elo", "source"]])
print(f"Catálogo experimental: {len(catalog_df)} jugadores con ELO válido")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.histplot(data=catalog_df, x="elo", bins=30, ax=axes[0], color="tab:blue")
axes[0].set_title("Catálogo candidato para pools experimentales")
axes[0].set_xlabel("ELO")
axes[0].set_ylabel("Jugadores")

sns.scatterplot(data=original_pool_df, x="elo", y=np.zeros(len(original_pool_df)), ax=axes[1], s=100)
axes[1].set_title("Pool original N=10 del notebook 120")
axes[1].set_xlabel("ELO")
axes[1].set_yticks([])
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 4) Embeddings, memory banks y predicción ELO

Para cada seed se crea un banco de memoria con `10` jugadores base. Cada jugador base aporta `memory_games_per_player` partidas embebidas con el modelo 120. Una query se predice comparando sus embeddings contra ese banco.

La predicción primaria de ELO es post-hoc: se toma la distancia agregada por jugador de memoria y se calcula una media ponderada por distancia de los ELO de los jugadores base más cercanos.


Desde esta versión se exportan también los estimadores compatibles con `121` y `122_chess_siamese_elo_open_scale`: vecino ejemplar, kNN de ejemplares, memoria por jugador y centroides. La columna `pred_elo_raw` se define como `player_memory_top1`, no como promedio ponderado, para evitar que el scatter oficial quede artificialmente comprimido hacia el ELO medio del banco. El promedio ponderado se conserva en `pred_elo_player_memory_weighted`.


In [ ]:
def cache_key(player: str, n_games: int, seed: int) -> str:
    payload = f"{player}|{n_games}|{seed}|{CONFIG.sequence_len}|{CONFIG.fullmove_start}|{CONFIG.fullmove_end}"
    return hashlib.md5(payload.encode("utf-8")).hexdigest()


def embed_player_games(
    player_row: pd.Series,
    n_games: int,
    seed: int,
    use_cache: bool = True,
) -> Tuple[pd.DataFrame, np.ndarray]:
    player_name = str(player_row["player"])
    zst_path = Path(player_row["zst_path"])
    elo = float(player_row["elo"])
    key = cache_key(player_name, n_games, seed)
    npz_path = EMBED_CACHE_DIR / f"{key}.npz"
    meta_path = EMBED_CACHE_DIR / f"{key}.csv"

    if use_cache and npz_path.exists() and meta_path.exists():
        meta_df = pd.read_csv(meta_path)
        embeddings = np.load(npz_path)["embeddings"].astype(np.float32)
        return meta_df, embeddings

    sampled_games, scanned_total, useful_seen = sample_random_games_from_zst(
        zst_path=zst_path,
        player_name=player_name,
        sample_size=n_games,
        seed=seed,
        max_games_to_scan=CONFIG.max_games_to_scan_per_player,
    )

    board_batches = []
    heat_batches = []
    meta_rows = []
    for local_rank, (game_index, game) in enumerate(sampled_games, start=1):
        try:
            boards, heats, side = build_single_game_tensor(
                game=game,
                player_name=player_name,
                sequence_len=CONFIG.sequence_len,
                fullmove_start=CONFIG.fullmove_start,
                fullmove_end=CONFIG.fullmove_end,
                image_size=CONFIG.image_size,
            )
        except Exception:
            continue

        board_batches.append(boards)
        heat_batches.append(heats)
        meta_rows.append(
            {
                "player": player_name,
                "elo": elo,
                "game_index": int(game_index),
                "local_game_rank": int(local_rank),
                "side": side,
                "scanned_total_games_in_file": int(scanned_total),
                "useful_games_seen_in_file": int(useful_seen),
            }
        )

    if not meta_rows:
        return pd.DataFrame(), np.empty((0, CONFIG.embedding_dim), dtype=np.float32)

    boards_np = np.stack(board_batches).astype(np.float32)
    heats_np = np.stack(heat_batches).astype(np.float32)
    embeddings = embedding_model.predict(
        {"board_sequence": boards_np, "heat_sequence": heats_np},
        batch_size=min(CONFIG.infer_batch_size, len(meta_rows)),
        verbose=0,
    ).astype(np.float32)

    meta_df = pd.DataFrame(meta_rows)
    if use_cache:
        np.savez_compressed(npz_path, embeddings=embeddings)
        meta_df.to_csv(meta_path, index=False)

    del boards_np, heats_np, board_batches, heat_batches
    gc.collect()
    return meta_df, embeddings


def pairwise_distance_matrix(query_embeddings: np.ndarray, ref_embeddings: np.ndarray) -> np.ndarray:
    query = np.asarray(query_embeddings, dtype=np.float32)
    reference = np.asarray(ref_embeddings, dtype=np.float32)
    sims = np.clip(query @ reference.T, -1.0, 1.0)
    dist_sq = np.maximum(2.0 - 2.0 * sims, 0.0)
    return np.sqrt(dist_sq + 1e-12).astype(np.float32)


def aggregate_player_distances(
    query_to_ref_dists: np.ndarray,
    ref_labels: np.ndarray,
    num_players: int,
    top_k: int,
) -> np.ndarray:
    out = np.full((query_to_ref_dists.shape[0], num_players), np.inf, dtype=np.float32)
    for player_idx in range(num_players):
        class_mask = ref_labels == player_idx
        class_dists = query_to_ref_dists[:, class_mask]
        if class_dists.size == 0:
            continue
        k = min(max(1, int(top_k)), class_dists.shape[1])
        topk = np.partition(class_dists, kth=k - 1, axis=1)[:, :k]
        out[:, player_idx] = topk.mean(axis=1)
    return out


def weighted_average_from_distances(
    values: np.ndarray,
    dists: np.ndarray,
    k: int,
    eps: float = 1e-6,
) -> float:
    values = np.asarray(values, dtype=np.float32)
    dists = np.asarray(dists, dtype=np.float32)
    valid = np.isfinite(values) & np.isfinite(dists)
    values = values[valid]
    dists = dists[valid]
    if values.size == 0:
        return float("nan")
    k = min(max(1, int(k)), values.size)
    order = np.argsort(dists)[:k]
    picked_values = values[order]
    picked_dists = dists[order]
    weights = 1.0 / np.maximum(picked_dists, eps)
    return float(np.sum(weights * picked_values) / np.maximum(np.sum(weights), eps))


def build_memory_bank(base_pool_df: pd.DataFrame, seed: int) -> Dict[str, object]:
    meta_frames = []
    emb_chunks = []
    label_chunks = []

    base_players = base_pool_df.sort_values("elo").reset_index(drop=True)
    player_to_label = {p: i for i, p in enumerate(base_players["player"].astype(str))}

    for _, row in base_players.iterrows():
        player = str(row["player"])
        print(f"  memoria seed={seed} | {player} (ELO {int(row['elo'])})")
        player_seed = stable_int_seed("memory", seed, player)
        meta_df, emb = embed_player_games(
            row,
            n_games=CONFIG.memory_games_per_player,
            seed=player_seed,
            use_cache=True,
        )
        if meta_df.empty:
            continue
        label = player_to_label[player]
        meta_df = meta_df.copy()
        meta_df["memory_seed"] = int(seed)
        meta_df["label"] = int(label)
        meta_frames.append(meta_df)
        emb_chunks.append(emb)
        label_chunks.append(np.full(len(emb), label, dtype=np.int32))

    if not emb_chunks:
        raise RuntimeError(f"No se pudo construir memory bank para seed={seed}")

    memory_meta_df = pd.concat(meta_frames, ignore_index=True)
    memory_embeddings = np.vstack(emb_chunks).astype(np.float32)
    memory_labels = np.concatenate(label_chunks).astype(np.int32)
    player_elos = base_players.set_index("player").loc[
        list(player_to_label.keys()), "elo"
    ].astype(float).values.astype(np.float32)

    centroid_matrix = []
    for label in range(len(player_elos)):
        class_emb = memory_embeddings[memory_labels == label]
        if class_emb.size == 0:
            centroid_matrix.append(np.full((CONFIG.embedding_dim,), np.nan, dtype=np.float32))
            continue
        centroid = class_emb.mean(axis=0).astype(np.float32)
        centroid /= np.clip(np.linalg.norm(centroid), 1e-8, None)
        centroid_matrix.append(centroid)
    centroid_matrix = np.vstack(centroid_matrix).astype(np.float32)

    return {
        "base_players": base_players,
        "player_to_label": player_to_label,
        "label_to_player": {v: k for k, v in player_to_label.items()},
        "player_elos": player_elos,
        "memory_meta_df": memory_meta_df,
        "memory_embeddings": memory_embeddings,
        "memory_labels": memory_labels,
        "reference_game_elos": memory_meta_df["elo"].astype(float).values.astype(np.float32),
        "centroid_matrix": centroid_matrix,
    }


def predict_elo_from_memory(query_embeddings: np.ndarray, memory_bank: Dict[str, object]) -> pd.DataFrame:
    memory_embeddings = memory_bank["memory_embeddings"]
    memory_labels = memory_bank["memory_labels"]
    player_elos = memory_bank["player_elos"]
    label_to_player = memory_bank["label_to_player"]
    reference_game_elos = memory_bank["reference_game_elos"]
    centroid_matrix = memory_bank["centroid_matrix"]

    q2ref = pairwise_distance_matrix(query_embeddings, memory_embeddings)

    exemplar_top1_idx = np.argmin(q2ref, axis=1)
    exemplar_top1_elo = reference_game_elos[exemplar_top1_idx]
    exemplar_top1_dist = q2ref[np.arange(len(query_embeddings)), exemplar_top1_idx]
    exemplar_knn_elo = np.array(
        [
            weighted_average_from_distances(
                values=reference_game_elos,
                dists=row_dists,
                k=CONFIG.exemplar_knn_k,
                eps=CONFIG.weight_epsilon,
            )
            for row_dists in q2ref
        ],
        dtype=np.float32,
    )

    player_dists = aggregate_player_distances(
        query_to_ref_dists=q2ref,
        ref_labels=memory_labels,
        num_players=len(player_elos),
        top_k=CONFIG.player_memory_top_k,
    )
    player_top1_idx = np.argmin(player_dists, axis=1)
    player_top1_dist = player_dists[np.arange(len(query_embeddings)), player_top1_idx]
    player_top1_elo = player_elos[player_top1_idx]
    player_weighted_elo = np.array(
        [
            weighted_average_from_distances(
                values=player_elos,
                dists=row_dists,
                k=CONFIG.player_regression_k,
                eps=CONFIG.weight_epsilon,
            )
            for row_dists in player_dists
        ],
        dtype=np.float32,
    )

    centroid_dists = pairwise_distance_matrix(query_embeddings, centroid_matrix)
    centroid_top1_idx = np.argmin(centroid_dists, axis=1)
    centroid_top1_elo = player_elos[centroid_top1_idx]
    centroid_top1_dist = centroid_dists[np.arange(len(query_embeddings)), centroid_top1_idx]
    centroid_weighted_elo = np.array(
        [
            weighted_average_from_distances(
                values=player_elos,
                dists=row_dists,
                k=CONFIG.centroid_regression_k,
                eps=CONFIG.weight_epsilon,
            )
            for row_dists in centroid_dists
        ],
        dtype=np.float32,
    )

    topk_player_order = np.argsort(player_dists, axis=1)[:, : min(CONFIG.player_regression_k, len(player_elos))]
    topk_player_names = [
        ", ".join(label_to_player[int(idx)] for idx in row)
        for row in topk_player_order
    ]
    topk_player_elos = [
        ", ".join(f"{float(player_elos[int(idx)]):.1f}" for idx in row)
        for row in topk_player_order
    ]
    topk_player_dists = [
        ", ".join(f"{float(player_dists[row_idx, int(idx)]):.6f}" for idx in row)
        for row_idx, row in enumerate(topk_player_order)
    ]

    return pd.DataFrame(
        {
            "pred_elo_exemplar_top1": exemplar_top1_elo,
            "pred_elo_exemplar_knn": exemplar_knn_elo,
            "pred_elo_player_memory_top1": player_top1_elo,
            "pred_elo_player_memory_weighted": player_weighted_elo,
            "pred_elo_centroid_top1": centroid_top1_elo,
            "pred_elo_centroid_weighted": centroid_weighted_elo,
            "dist_exemplar_top1": exemplar_top1_dist,
            "dist_player_memory_top1": player_top1_dist,
            "dist_centroid_top1": centroid_top1_dist,
            "pred_player_memory_top1": [label_to_player[int(idx)] for idx in player_top1_idx],
            "pred_centroid_top1": [label_to_player[int(idx)] for idx in centroid_top1_idx],
            "pred_player_exemplar_top1": [label_to_player[int(memory_labels[idx])] for idx in exemplar_top1_idx],
            "pred_game_exemplar_top1": memory_bank["memory_meta_df"].iloc[exemplar_top1_idx]["game_index"].astype(int).values,
            # Compatibilidad con el scatter de 121/122: raw = top-1 por memoria de jugador.
            "pred_elo_top1": player_top1_elo,
            "pred_elo_raw": player_top1_elo,
            "pred_elo_raw_weighted_k5": player_weighted_elo,
            "top1_distance": player_top1_dist,
            "pred_player_top1": [label_to_player[int(idx)] for idx in player_top1_idx],
            "topk_player_names": topk_player_names,
            "topk_player_elos": topk_player_elos,
            "topk_player_dists": topk_player_dists,
            "player_dist_min": np.min(player_dists, axis=1),
            "player_dist_max": np.max(player_dists, axis=1),
            "player_dist_std": np.std(player_dists, axis=1),
        }
    )


## A2) Repetibilidad: 5 seeds, 5 bancos de memoria

Cada seed genera un pool base distinto de `N=10`, cubierto por ELO. El modelo no se entrena de nuevo: solo se recalculan los embeddings de referencia y se evalúan jugadores-query no incluidos en el banco.


In [ ]:
def evaluate_query_pool_for_seed(
    seed: int,
    query_pool_df: pd.DataFrame,
    memory_bank: Dict[str, object],
) -> pd.DataFrame:
    pred_csv_path = SEED_DIR / f"seed_{seed}_game_predictions.csv"
    if CONFIG.resume_seed_predictions and pred_csv_path.exists():
        cached = pd.read_csv(pred_csv_path)
        required_cols = {
            "pred_elo_exemplar_top1",
            "pred_elo_exemplar_knn",
            "pred_elo_player_memory_top1",
            "pred_elo_player_memory_weighted",
            "pred_elo_centroid_top1",
            "pred_elo_centroid_weighted",
            "pred_elo_raw",
        }
        if not cached.empty and required_cols.issubset(cached.columns):
            print(f"Seed {seed}: usando predicciones cacheadas en {pred_csv_path}")
            return cached
        print(f"Seed {seed}: cache antigua/incompleta; se regenera {pred_csv_path}")
        pred_csv_path.unlink(missing_ok=True)

    rows = []
    for idx, (_, row) in enumerate(query_pool_df.iterrows(), start=1):
        player = str(row["player"])
        print(f"  query seed={seed} [{idx}/{len(query_pool_df)}] {player} (ELO {int(row['elo'])})")
        player_seed = stable_int_seed("query", seed, player)
        meta_df, emb = embed_player_games(
            row,
            n_games=CONFIG.query_games_per_player,
            seed=player_seed,
            use_cache=True,
        )
        if meta_df.empty:
            continue

        pred_df = predict_elo_from_memory(emb, memory_bank)
        player_df = pd.concat([meta_df.reset_index(drop=True), pred_df], axis=1)
        player_df = player_df.rename(columns={"player": "query_player", "elo": "query_elo"})
        player_df["seed"] = int(seed)
        rows.append(player_df)

    if not rows:
        raise RuntimeError(f"No se generaron predicciones para seed={seed}")

    out = pd.concat(rows, ignore_index=True)
    out.to_csv(pred_csv_path, index=False)
    return out


def split_calibration_test(players: List[str], seed: int) -> Tuple[set, set]:
    players = sorted(players)
    rng = random.Random(stable_int_seed("calibration", seed))
    shuffled = players.copy()
    rng.shuffle(shuffled)
    n_cal = max(2, int(round(len(shuffled) * CONFIG.calibration_fraction)))
    n_cal = min(n_cal, max(1, len(shuffled) - 2))
    cal_players = set(shuffled[:n_cal])
    test_players = set(shuffled[n_cal:])
    return cal_players, test_players


def fit_linear_correction(cal_df: pd.DataFrame, pred_col: str = "pred_elo_raw_mean") -> Tuple[float, float]:
    valid = cal_df.dropna(subset=[pred_col, "query_elo"]).copy()
    if len(valid) < 2 or valid[pred_col].nunique() < 2:
        return 1.0, 0.0
    slope, intercept = np.polyfit(valid[pred_col].values, valid["query_elo"].values, deg=1)
    return float(slope), float(intercept)


PREDICTION_METHOD_COLUMNS = [
    "pred_elo_exemplar_top1",
    "pred_elo_exemplar_knn",
    "pred_elo_player_memory_top1",
    "pred_elo_player_memory_weighted",
    "pred_elo_centroid_top1",
    "pred_elo_centroid_weighted",
    "pred_elo_raw",
]


def summarize_seed_predictions(game_df: pd.DataFrame, seed: int) -> pd.DataFrame:
    metric_rows = []
    players = game_df["query_player"].drop_duplicates().astype(str).tolist()
    cal_players, test_players = split_calibration_test(players, seed)

    for nq in CONFIG.n_query_grid:
        limited = (
            game_df.sort_values(["query_player", "local_game_rank"])
            .groupby("query_player", group_keys=False)
            .head(int(nq))
            .copy()
        )

        agg_spec = {
            "games_used": ("game_index", "count"),
            "top1_distance_mean": ("top1_distance", "mean"),
        }
        for col in PREDICTION_METHOD_COLUMNS:
            if col in limited.columns:
                agg_spec[f"{col}_mean"] = (col, "mean")

        player_df = (
            limited.groupby(["query_player", "query_elo"], as_index=False)
            .agg(**agg_spec)
            .loc[lambda df: df["games_used"] >= min(nq, CONFIG.query_games_per_player)]
            .copy()
        )

        cal_df = player_df.loc[player_df["query_player"].isin(cal_players)].copy()
        test_df = player_df.loc[player_df["query_player"].isin(test_players)].copy()
        slope, intercept = fit_linear_correction(cal_df, pred_col="pred_elo_raw_mean")
        test_df["pred_elo_corrected"] = slope * test_df["pred_elo_raw_mean"] + intercept

        for split_name, split_df in [("calibration", cal_df), ("test", test_df)]:
            if split_name == "calibration":
                split_df = split_df.copy()
                split_df["pred_elo_corrected"] = slope * split_df["pred_elo_raw_mean"] + intercept
            valid = split_df.dropna(subset=["pred_elo_raw_mean", "query_elo"]).copy()
            if valid.empty:
                continue

            row = {
                "seed": int(seed),
                "nq": int(nq),
                "split": split_name,
                "players": int(len(valid)),
                "linear_slope": slope,
                "linear_intercept": intercept,
            }
            for col in PREDICTION_METHOD_COLUMNS:
                mean_col = f"{col}_mean"
                if mean_col not in valid.columns:
                    continue
                pred = valid[mean_col]
                row[f"mae__{col}"] = float(np.mean(np.abs(pred - valid["query_elo"])))
                row[f"rmse__{col}"] = float(np.sqrt(np.mean((pred - valid["query_elo"]) ** 2)))
                row[f"corr__{col}"] = (
                    float(np.corrcoef(valid["query_elo"], pred)[0, 1])
                    if len(valid) >= 2 and pred.nunique() > 1
                    else np.nan
                )

            corrected_errors = np.abs(valid["pred_elo_corrected"] - valid["query_elo"])
            row["mae_top1"] = row.get("mae__pred_elo_player_memory_top1", np.nan)
            row["mae_raw"] = row.get("mae__pred_elo_raw", np.nan)
            row["mae_weighted_k5"] = row.get("mae__pred_elo_player_memory_weighted", np.nan)
            row["mae_corrected"] = float(np.mean(corrected_errors))
            row["rmse_raw"] = row.get("rmse__pred_elo_raw", np.nan)
            row["rmse_corrected"] = float(np.sqrt(np.mean((valid["pred_elo_corrected"] - valid["query_elo"]) ** 2)))
            row["corr_raw"] = row.get("corr__pred_elo_raw", np.nan)
            metric_rows.append(row)

    return pd.DataFrame(metric_rows)

def run_seed(seed: int) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    print("=" * 90)
    print(f"SEED {seed}")
    print("=" * 90)

    base_pool_df = select_elo_covered_pool(
        catalog_df=catalog_df,
        n=CONFIG.base_pool_size,
        seed=seed,
    )
    query_pool_df = select_elo_covered_pool(
        catalog_df=catalog_df,
        n=CONFIG.query_players_per_seed,
        seed=seed + 10_000,
        exclude_players=base_pool_df["player"].tolist(),
    )

    base_pool_df = base_pool_df.copy()
    query_pool_df = query_pool_df.copy()
    base_pool_df["seed"] = int(seed)
    query_pool_df["seed"] = int(seed)

    display(base_pool_df[["seed", "player", "elo", "total_games"]])

    memory_bank = build_memory_bank(base_pool_df, seed=seed)
    game_df = evaluate_query_pool_for_seed(seed, query_pool_df, memory_bank)
    metrics_df = summarize_seed_predictions(game_df, seed=seed)

    return base_pool_df, query_pool_df, game_df, metrics_df


all_base_pools = []
all_query_pools = []
all_metrics = []

for seed in CONFIG.seeds:
    base_pool_df, query_pool_df, game_df, metrics_df = run_seed(seed)
    all_base_pools.append(base_pool_df)
    all_query_pools.append(query_pool_df)
    all_metrics.append(metrics_df)

base_pools_df = pd.concat(all_base_pools, ignore_index=True)
query_pools_df = pd.concat(all_query_pools, ignore_index=True)
metrics_df = pd.concat(all_metrics, ignore_index=True)

base_pools_df.to_csv(BASE_POOLS_CSV_PATH, index=False)
query_pools_df.to_csv(QUERY_POOLS_CSV_PATH, index=False)
metrics_df.to_csv(A2_METRICS_CSV_PATH, index=False)

run_meta = {
    "config": asdict(CONFIG),
    "event_dir": str(EVENT_DIR),
    "checkpoint_path": str(CHECKPOINT_PATH),
    "index_db": str(INDEX_DB),
    "players_dir": str(ZST_PLAYERS_DIR),
    "original_120_pool": original_pool_df.to_dict("records"),
}
RUN_META_PATH.write_text(json.dumps(run_meta, ensure_ascii=False, indent=2), encoding="utf-8")

display(metrics_df.head(20))


## A2) Resumen de repetibilidad

El resumen principal se toma sobre el split `test` y `nq=20`, porque usa el máximo de partidas-query disponible por jugador y mide generalización de la corrección lineal sobre jugadores no usados para calibrar.


In [ ]:
a2_test_df = metrics_df.loc[
    (metrics_df["split"] == "test") & (metrics_df["nq"] == max(CONFIG.n_query_grid))
].copy()

a2_summary_df = pd.DataFrame(
    [
        {
            "n_seeds": int(a2_test_df["seed"].nunique()),
            "nq": int(max(CONFIG.n_query_grid)),
            "mae_raw_mean": float(a2_test_df["mae_raw"].mean()),
            "mae_raw_std": float(a2_test_df["mae_raw"].std(ddof=1)),
            "mae_corrected_mean": float(a2_test_df["mae_corrected"].mean()),
            "mae_corrected_std": float(a2_test_df["mae_corrected"].std(ddof=1)),
            "corr_raw_mean": float(a2_test_df["corr_raw"].mean()),
        }
    ]
)

display(a2_test_df[["seed", "players", "mae_raw", "mae_corrected", "corr_raw", "linear_slope", "linear_intercept"]])
display(a2_summary_df)

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = a2_test_df.melt(
    id_vars=["seed"],
    value_vars=["mae_raw", "mae_corrected"],
    var_name="metric",
    value_name="mae",
)
sns.barplot(data=plot_df, x="seed", y="mae", hue="metric", ax=ax)
ax.set_title("A2 · MAE por seed con nq=20")
ax.set_xlabel("Seed")
ax.set_ylabel("MAE ELO")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


## A3) Número de partidas-query necesarias

Para cada `nq`, se promedian las predicciones de las primeras `nq` partidas válidas de cada jugador-query y se calcula MAE. Se reporta tanto el MAE crudo como el MAE tras corrección lineal calibrada en jugadores separados.


In [ ]:
a3_df = metrics_df.loc[metrics_df["split"] == "test"].copy()
a3_summary_df = (
    a3_df.groupby("nq", as_index=False)
    .agg(
        seeds=("seed", "nunique"),
        mae_raw_mean=("mae_raw", "mean"),
        mae_raw_std=("mae_raw", "std"),
        mae_corrected_mean=("mae_corrected", "mean"),
        mae_corrected_std=("mae_corrected", "std"),
        corr_raw_mean=("corr_raw", "mean"),
        players_mean=("players", "mean"),
    )
    .sort_values("nq")
    .reset_index(drop=True)
)
a3_summary_df.to_csv(A3_METRICS_CSV_PATH, index=False)

display(a3_summary_df)

fig, ax = plt.subplots(figsize=(10, 6))
ax.errorbar(
    a3_summary_df["nq"],
    a3_summary_df["mae_raw_mean"],
    yerr=a3_summary_df["mae_raw_std"],
    marker="o",
    capsize=4,
    label="MAE crudo",
)
ax.errorbar(
    a3_summary_df["nq"],
    a3_summary_df["mae_corrected_mean"],
    yerr=a3_summary_df["mae_corrected_std"],
    marker="o",
    capsize=4,
    label="MAE corregido linealmente",
)
ax.set_title("A3 · MAE vs número de partidas-query")
ax.set_xlabel("nq: partidas del jugador-query")
ax.set_ylabel("MAE ELO")
ax.set_xticks(list(CONFIG.n_query_grid))
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## Corrección lineal final

La calibración lineal ajusta una recta entre el ELO estimado por similitud y el ELO real en un subconjunto de calibración:

`ELO_real ≈ a · ELO_predicho + b`

Luego esa recta se aplica sobre jugadores de test. Esta corrección es razonable si la nube `ELO real` vs `ELO predicho` muestra relación aproximadamente lineal pero con sesgo de escala o desplazamiento.


In [ ]:
best_nq = max(CONFIG.n_query_grid)
calibration_lines_df = metrics_df.loc[
    (metrics_df["split"] == "test") & (metrics_df["nq"] == best_nq),
    ["seed", "linear_slope", "linear_intercept", "mae_raw", "mae_corrected"],
].copy()
calibration_lines_df["mae_delta"] = calibration_lines_df["mae_corrected"] - calibration_lines_df["mae_raw"]

display(calibration_lines_df)

print("Interpretación rápida:")
print("- mae_delta < 0 implica que la corrección lineal mejora el MAE en test.")
print("- slope corrige compresión/expansión de escala.")
print("- intercept corrige desplazamiento global del estimador.")


In [ ]:
DIAGNOSTICS_DIR = EVAL_DIR / "diagnostics"
DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

SCATTER_TEST_CSV_PATH = EVAL_DIR / "scatter_data_test.csv"
PLAYER_NQ_SUMMARY_CSV_PATH = EVAL_DIR / "player_nq_summary_all_splits.csv"
COLLAPSE_DIAGNOSTICS_CSV_PATH = EVAL_DIR / "prediction_collapse_diagnostics.csv"
RANDOM_PLAYERS_CSV_PATH = EVAL_DIR / "random_individual_players_by_seed.csv"
ALL_GAME_PREDICTIONS_CSV_PATH = EVAL_DIR / "all_seed_game_predictions.csv"


def load_all_seed_predictions() -> pd.DataFrame:
    frames = []
    for path in sorted(SEED_DIR.glob("seed_*_game_predictions.csv")):
        frame = pd.read_csv(path)
        if frame.empty:
            continue
        if "seed" not in frame.columns:
            seed_token = path.stem.split("_")[1]
            frame["seed"] = int(seed_token)
        frames.append(frame)
    if not frames:
        raise RuntimeError("No hay CSV en seed_predictions/. Ejecuta antes la sección A2.")
    out = pd.concat(frames, ignore_index=True)
    out.to_csv(ALL_GAME_PREDICTIONS_CSV_PATH, index=False)
    return out


def build_player_nq_summary_from_games(game_predictions_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for seed in sorted(game_predictions_df["seed"].dropna().astype(int).unique()):
        seed_games = game_predictions_df.loc[game_predictions_df["seed"].astype(int) == seed].copy()
        players = seed_games["query_player"].drop_duplicates().astype(str).tolist()
        cal_players, test_players = split_calibration_test(players, seed)

        for nq in CONFIG.n_query_grid:
            limited = (
                seed_games.sort_values(["query_player", "local_game_rank"])
                .groupby("query_player", group_keys=False)
                .head(int(nq))
                .copy()
            )
            player_df = (
                limited.groupby(["query_player", "query_elo"], as_index=False)
                .agg(
                    games_used=("game_index", "count"),
                    pred_elo_top1_mean=("pred_elo_top1", "mean"),
                    pred_elo_raw_mean=("pred_elo_raw", "mean"),
                    pred_elo_raw_std=("pred_elo_raw", "std"),
                    pred_elo_raw_min=("pred_elo_raw", "min"),
                    pred_elo_raw_max=("pred_elo_raw", "max"),
                    pred_elo_player_memory_weighted_mean=("pred_elo_player_memory_weighted", "mean"),
                    pred_elo_exemplar_top1_mean=("pred_elo_exemplar_top1", "mean"),
                    pred_elo_exemplar_knn_mean=("pred_elo_exemplar_knn", "mean"),
                    pred_elo_centroid_top1_mean=("pred_elo_centroid_top1", "mean"),
                    pred_elo_centroid_weighted_mean=("pred_elo_centroid_weighted", "mean"),
                    top1_distance_mean=("top1_distance", "mean"),
                    pred_player_top1_mode=("pred_player_top1", lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan),
                    pred_player_top1_unique=("pred_player_top1", "nunique"),
                )
                .loc[lambda df: df["games_used"] >= min(nq, CONFIG.query_games_per_player)]
                .copy()
            )
            metric_row = metrics_df.loc[
                (metrics_df["seed"].astype(int) == seed)
                & (metrics_df["nq"].astype(int) == int(nq))
                & (metrics_df["split"] == "test")
            ]
            if metric_row.empty:
                slope, intercept = fit_linear_correction(
                    player_df.loc[player_df["query_player"].isin(cal_players)].copy(),
                    pred_col="pred_elo_raw_mean",
                )
            else:
                slope = float(metric_row.iloc[0]["linear_slope"])
                intercept = float(metric_row.iloc[0]["linear_intercept"])

            player_df["seed"] = int(seed)
            player_df["nq"] = int(nq)
            player_df["split"] = np.where(
                player_df["query_player"].isin(cal_players),
                "calibration",
                "test",
            )
            player_df["linear_slope"] = slope
            player_df["linear_intercept"] = intercept
            player_df["pred_elo_corrected"] = slope * player_df["pred_elo_raw_mean"] + intercept
            player_df["abs_error_raw"] = (player_df["pred_elo_raw_mean"] - player_df["query_elo"]).abs()
            player_df["abs_error_corrected"] = (player_df["pred_elo_corrected"] - player_df["query_elo"]).abs()
            for method_col in [
                "pred_elo_player_memory_weighted_mean",
                "pred_elo_exemplar_top1_mean",
                "pred_elo_exemplar_knn_mean",
                "pred_elo_centroid_top1_mean",
                "pred_elo_centroid_weighted_mean",
            ]:
                if method_col in player_df.columns:
                    player_df[f"abs_error__{method_col}"] = (player_df[method_col] - player_df["query_elo"]).abs()
            player_df["raw_minus_true"] = player_df["pred_elo_raw_mean"] - player_df["query_elo"]
            player_df["corrected_minus_true"] = player_df["pred_elo_corrected"] - player_df["query_elo"]
            rows.append(player_df)

    out = pd.concat(rows, ignore_index=True)
    out = out.sort_values(["seed", "nq", "split", "query_elo", "query_player"]).reset_index(drop=True)
    out.to_csv(PLAYER_NQ_SUMMARY_CSV_PATH, index=False)
    return out


all_game_predictions_df = load_all_seed_predictions()
player_nq_summary_df = build_player_nq_summary_from_games(all_game_predictions_df)

scatter_data_test_df = player_nq_summary_df.loc[
    (player_nq_summary_df["split"] == "test")
    & (player_nq_summary_df["nq"] == max(CONFIG.n_query_grid))
].copy()
scatter_data_test_df.to_csv(SCATTER_TEST_CSV_PATH, index=False)

collapse_rows = []
for seed, seed_df in scatter_data_test_df.groupby("seed"):
    game_seed_df = all_game_predictions_df.loc[all_game_predictions_df["seed"].astype(int) == int(seed)].copy()
    pred_range = float(seed_df["pred_elo_raw_mean"].max() - seed_df["pred_elo_raw_mean"].min())
    true_range = float(seed_df["query_elo"].max() - seed_df["query_elo"].min())
    collapse_rows.append(
        {
            "seed": int(seed),
            "players_test": int(seed_df["query_player"].nunique()),
            "games": int(len(game_seed_df)),
            "true_elo_min": float(seed_df["query_elo"].min()),
            "true_elo_max": float(seed_df["query_elo"].max()),
            "true_elo_std": float(seed_df["query_elo"].std(ddof=1)),
            "pred_raw_min": float(seed_df["pred_elo_raw_mean"].min()),
            "pred_raw_max": float(seed_df["pred_elo_raw_mean"].max()),
            "pred_raw_range": pred_range,
            "pred_raw_std": float(seed_df["pred_elo_raw_mean"].std(ddof=1)),
            "range_ratio_pred_over_true": pred_range / true_range if true_range > 0 else np.nan,
            "top1_unique_players_game_level": int(game_seed_df["pred_player_top1"].nunique()),
            "top1_mode_game_level": game_seed_df["pred_player_top1"].mode().iloc[0] if not game_seed_df["pred_player_top1"].mode().empty else np.nan,
            "top1_mode_share_game_level": float(game_seed_df["pred_player_top1"].value_counts(normalize=True).iloc[0]),
        }
    )

collapse_diagnostics_df = pd.DataFrame(collapse_rows).sort_values("seed").reset_index(drop=True)
collapse_diagnostics_df.to_csv(COLLAPSE_DIAGNOSTICS_CSV_PATH, index=False)

print("Diagnóstico de colapso/compresión de predicción raw:")
display(collapse_diagnostics_df)

print("Scatter correcto para el split test exportado en:")
print(SCATTER_TEST_CSV_PATH)
display(scatter_data_test_df.head(20))


fig, axes = plt.subplots(len(CONFIG.seeds), 2, figsize=(15, 4 * len(CONFIG.seeds)), sharex=False, sharey=False)
if len(CONFIG.seeds) == 1:
    axes = np.array([axes])
for row_idx, seed in enumerate(sorted(scatter_data_test_df["seed"].unique())):
    seed_df = scatter_data_test_df.loc[scatter_data_test_df["seed"] == seed].copy()
    raw_ax = axes[row_idx, 0]
    corr_ax = axes[row_idx, 1]
    raw_ax.scatter(seed_df["query_elo"], seed_df["pred_elo_raw_mean"], alpha=0.75, s=45)
    corr_ax.scatter(seed_df["query_elo"], seed_df["pred_elo_corrected"], alpha=0.75, s=45, color="tab:orange")
    lo = float(min(seed_df["query_elo"].min(), seed_df["pred_elo_raw_mean"].min(), seed_df["pred_elo_corrected"].min()))
    hi = float(max(seed_df["query_elo"].max(), seed_df["pred_elo_raw_mean"].max(), seed_df["pred_elo_corrected"].max()))
    raw_ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    corr_ax.plot([lo, hi], [lo, hi], "k--", linewidth=1)
    raw_ax.set_title(f"Seed {seed} · raw")
    corr_ax.set_title(f"Seed {seed} · corrected")
    raw_ax.set_xlabel("ELO real")
    corr_ax.set_xlabel("ELO real")
    raw_ax.set_ylabel("ELO predicho raw")
    corr_ax.set_ylabel("ELO predicho corregido")
    raw_ax.grid(True, alpha=0.3)
    corr_ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


sample_rows = []
for seed, seed_df in scatter_data_test_df.groupby("seed"):
    rng = random.Random(stable_int_seed("individual_examples", int(seed)))
    players = sorted(seed_df["query_player"].astype(str).unique())
    picked = rng.sample(players, k=min(5, len(players)))
    for player in picked:
        sample_rows.append({"seed": int(seed), "query_player": player})
random_players_df = pd.DataFrame(sample_rows)
random_players_df.to_csv(RANDOM_PLAYERS_CSV_PATH, index=False)

individual_df = player_nq_summary_df.merge(random_players_df, on=["seed", "query_player"], how="inner")
individual_df.to_csv(DIAGNOSTICS_DIR / "random_individual_player_nq_curves.csv", index=False)

display(random_players_df)

for seed in sorted(individual_df["seed"].unique()):
    seed_individual_df = individual_df.loc[individual_df["seed"] == seed].copy()
    fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=False)
    if len(seed_individual_df["query_player"].unique()) < 5:
        axes = np.atleast_1d(axes)
    for ax, (player, player_df) in zip(axes, seed_individual_df.groupby("query_player")):
        player_df = player_df.sort_values("nq")
        true_elo = float(player_df["query_elo"].iloc[0])
        ax.plot(player_df["nq"], player_df["pred_elo_raw_mean"], marker="o", label="raw")
        ax.plot(player_df["nq"], player_df["pred_elo_corrected"], marker="o", label="corrected")
        ax.axhline(true_elo, color="black", linestyle="--", linewidth=1, label="real")
        ax.set_title(f"{player}\nELO real={true_elo:.0f}")
        ax.set_xlabel("nq")
        ax.set_ylabel("ELO")
        ax.set_xticks(list(CONFIG.n_query_grid))
        ax.grid(True, alpha=0.3)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    fig.suptitle(f"Seed {seed} · 5 jugadores test aleatorios: ELO estimado vs partidas usadas", y=1.08)
    plt.tight_layout()
    plt.show()


for seed in sorted(all_game_predictions_df["seed"].unique()):
    game_seed_df = all_game_predictions_df.loc[all_game_predictions_df["seed"] == seed].copy()
    counts = game_seed_df["pred_player_top1"].value_counts().reset_index()
    counts.columns = ["pred_player_top1", "games"]
    counts.to_csv(DIAGNOSTICS_DIR / f"seed_{int(seed)}_top1_player_usage.csv", index=False)
    plt.figure(figsize=(12, 4))
    ax = sns.barplot(data=counts, x="pred_player_top1", y="games", color="tab:blue")
    ax.set_title(f"Seed {int(seed)} · uso de jugadores ancla top-1")
    ax.set_xlabel("Jugador del banco predicho como top-1")
    ax.set_ylabel("Partidas query")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

print("Exportaciones adicionales:")
print(f"- {ALL_GAME_PREDICTIONS_CSV_PATH}")
print(f"- {PLAYER_NQ_SUMMARY_CSV_PATH}")
print(f"- {SCATTER_TEST_CSV_PATH}")
print(f"- {COLLAPSE_DIAGNOSTICS_CSV_PATH}")
print(f"- {RANDOM_PLAYERS_CSV_PATH}")
print(f"- {DIAGNOSTICS_DIR}")


## 122B) Experimentos de rescate del estimador ELO

La gráfica horizontal no implica por sí sola que el embedding sea inútil, pero sí muestra que el estimador `pred_elo_raw` está demasiado comprimido. Esta sección separa dos preguntas:

1. ¿El colapso viene de promediar demasiados anclas dentro de un banco pequeño (`N=10`, `k=5`)?
2. ¿Mejora al aumentar el número de jugadores base del banco sin reentrenar el modelo?

La primera comprobación usa los CSV ya generados. La segunda es opcional y puede tardar: crea bancos mayores con el mismo modelo 120, reutilizando cachés de embeddings cuando existan.


In [ ]:
RESCUE_DIR = EVAL_DIR / "rescue_experiments"
RESCUE_DIR.mkdir(parents=True, exist_ok=True)

RESCUE_EXISTING_ESTIMATORS_CSV_PATH = RESCUE_DIR / "existing_estimators_summary.csv"
RESCUE_MEMORY_SIZE_METRICS_CSV_PATH = RESCUE_DIR / "memory_size_metrics.csv"
RESCUE_MEMORY_SIZE_PLAYERS_CSV_PATH = RESCUE_DIR / "memory_size_player_predictions.csv"


def summarize_existing_estimators(player_nq_summary_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for nq in CONFIG.n_query_grid:
        subset = player_nq_summary_df.loc[
            (player_nq_summary_df["split"] == "test")
            & (player_nq_summary_df["nq"] == int(nq))
        ].copy()
        if subset.empty:
            continue
        for col, label in [
            ("pred_elo_top1_mean", "player_memory_top1/raw"),
            ("pred_elo_player_memory_weighted_mean", f"player_memory_weighted_k{CONFIG.player_regression_k}"),
            ("pred_elo_exemplar_top1_mean", "exemplar_top1"),
            ("pred_elo_exemplar_knn_mean", f"exemplar_knn_k{CONFIG.exemplar_knn_k}"),
            ("pred_elo_centroid_top1_mean", "centroid_top1"),
            ("pred_elo_centroid_weighted_mean", f"centroid_weighted_k{CONFIG.centroid_regression_k}"),
            ("pred_elo_corrected", "raw_corrected"),
        ]:
            if col not in subset.columns:
                continue
            valid = subset.dropna(subset=[col, "query_elo"]).copy()
            if valid.empty:
                continue
            pred_range = float(valid[col].max() - valid[col].min())
            true_range = float(valid["query_elo"].max() - valid["query_elo"].min())
            corr = (
                float(np.corrcoef(valid["query_elo"], valid[col])[0, 1])
                if len(valid) >= 2 and valid[col].nunique() > 1
                else np.nan
            )
            rows.append(
                {
                    "nq": int(nq),
                    "estimator": label,
                    "players": int(len(valid)),
                    "mae": float(np.mean(np.abs(valid[col] - valid["query_elo"]))),
                    "rmse": float(np.sqrt(np.mean((valid[col] - valid["query_elo"]) ** 2))),
                    "corr": corr,
                    "pred_std": float(valid[col].std(ddof=1)),
                    "pred_range": pred_range,
                    "true_range": true_range,
                    "range_ratio_pred_over_true": pred_range / true_range if true_range > 0 else np.nan,
                }
            )
    out = pd.DataFrame(rows)
    out.to_csv(RESCUE_EXISTING_ESTIMATORS_CSV_PATH, index=False)
    return out


existing_estimators_df = summarize_existing_estimators(player_nq_summary_df)
display(existing_estimators_df)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.lineplot(data=existing_estimators_df, x="nq", y="mae", hue="estimator", marker="o", ax=axes[0])
axes[0].set_title("Estimadores existentes · MAE vs nq")
axes[0].set_xlabel("nq")
axes[0].set_ylabel("MAE ELO")
axes[0].grid(True, alpha=0.3)

sns.lineplot(data=existing_estimators_df, x="nq", y="range_ratio_pred_over_true", hue="estimator", marker="o", ax=axes[1])
axes[1].set_title("Compresión de rango predicho")
axes[1].set_xlabel("nq")
axes[1].set_ylabel("rango predicho / rango real")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Prueba opcional: bancos de memoria más grandes sin reentrenar

Si `weighted_k5` queda horizontal con `N=10`, la prueba más justa es aumentar los jugadores ancla del banco manteniendo el modelo congelado. Esto no arregla el entrenamiento, pero sí comprueba si el fallo está en el estimador por falta de resolución del banco.

Por defecto `RUN_RESCUE_MEMORY_SIZE_EXPERIMENT = False` para no lanzar una evaluación costosa accidentalmente. Actívalo y ejecuta la celda si quieres producir el CSV comparativo.


In [ ]:
RUN_RESCUE_MEMORY_SIZE_EXPERIMENT = False
RESCUE_MEMORY_SIZES = (10, 25, 50, 100)
RESCUE_REGRESSION_KS = (1, 2, 3, 5)
RESCUE_SEEDS = CONFIG.seeds


def downsample_pool_by_elo(pool_df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    work = pool_df.sort_values("elo").reset_index(drop=True)
    if len(work) < n:
        raise RuntimeError(f"Pool insuficiente: {len(work)} < {n}")
    if len(work) == n:
        return work.copy()
    rng = random.Random(stable_int_seed("downsample_pool", seed, n))
    chunks = np.array_split(np.arange(len(work)), n)
    picked = []
    for chunk in chunks:
        picked.append(int(rng.choice(list(chunk))))
    return work.iloc[picked].sort_values("elo").reset_index(drop=True)


def load_query_embeddings_for_pool(query_pool_df: pd.DataFrame, seed: int) -> Tuple[pd.DataFrame, np.ndarray]:
    meta_frames = []
    emb_chunks = []
    for _, row in query_pool_df.iterrows():
        player = str(row["player"])
        player_seed = stable_int_seed("query", seed, player)
        meta_df, emb = embed_player_games(
            row,
            n_games=CONFIG.query_games_per_player,
            seed=player_seed,
            use_cache=True,
        )
        if meta_df.empty:
            continue
        meta_df = meta_df.rename(columns={"player": "query_player", "elo": "query_elo"})
        meta_df["seed"] = int(seed)
        meta_frames.append(meta_df)
        emb_chunks.append(emb)
    if not emb_chunks:
        raise RuntimeError(f"Sin embeddings query para seed={seed}")
    return pd.concat(meta_frames, ignore_index=True), np.vstack(emb_chunks).astype(np.float32)


def predict_elo_from_memory_with_regression_k(
    query_embeddings: np.ndarray,
    memory_bank: Dict[str, object],
    regression_k: int,
) -> pd.DataFrame:
    memory_embeddings = memory_bank["memory_embeddings"]
    memory_labels = memory_bank["memory_labels"]
    player_elos = memory_bank["player_elos"]
    label_to_player = memory_bank["label_to_player"]

    q2ref = pairwise_distance_matrix(query_embeddings, memory_embeddings)
    player_dists = aggregate_player_distances(
        query_to_ref_dists=q2ref,
        ref_labels=memory_labels,
        num_players=len(player_elos),
        top_k=CONFIG.player_memory_top_k,
    )
    top1_idx = np.argmin(player_dists, axis=1)
    pred_top1 = player_elos[top1_idx]
    pred_weighted = np.array(
        [
            weighted_average_from_distances(
                values=player_elos,
                dists=row_dists,
                k=regression_k,
                eps=CONFIG.weight_epsilon,
            )
            for row_dists in player_dists
        ],
        dtype=np.float32,
    )
    return pd.DataFrame(
        {
            "pred_elo_top1": pred_top1,
            "pred_elo_raw": pred_weighted,
            "pred_player_top1": [label_to_player[int(idx)] for idx in top1_idx],
            "top1_distance": player_dists[np.arange(len(query_embeddings)), top1_idx],
        }
    )


def evaluate_rescue_memory_sizes() -> Tuple[pd.DataFrame, pd.DataFrame]:
    metric_rows = []
    player_rows = []
    max_memory_size = max(RESCUE_MEMORY_SIZES)

    for seed in RESCUE_SEEDS:
        print("=" * 90)
        print(f"Rescue seed={seed}")
        print("=" * 90)
        max_base_pool_df = select_elo_covered_pool(
            catalog_df=catalog_df,
            n=max_memory_size,
            seed=int(seed),
        )
        query_pool_df = select_elo_covered_pool(
            catalog_df=catalog_df,
            n=CONFIG.query_players_per_seed,
            seed=int(seed) + 10_000,
            exclude_players=max_base_pool_df["player"].tolist(),
        )
        query_meta_df, query_embeddings = load_query_embeddings_for_pool(query_pool_df, int(seed))
        query_players = query_meta_df["query_player"].drop_duplicates().astype(str).tolist()
        cal_players, test_players = split_calibration_test(query_players, int(seed))

        for memory_size in RESCUE_MEMORY_SIZES:
            base_pool_df = downsample_pool_by_elo(max_base_pool_df, int(memory_size), int(seed))
            memory_bank = build_memory_bank(base_pool_df, seed=stable_int_seed("rescue_memory", seed, memory_size))
            for regression_k in RESCUE_REGRESSION_KS:
                pred_df = predict_elo_from_memory_with_regression_k(
                    query_embeddings=query_embeddings,
                    memory_bank=memory_bank,
                    regression_k=int(regression_k),
                )
                game_df = pd.concat([query_meta_df.reset_index(drop=True), pred_df], axis=1)
                limited = (
                    game_df.sort_values(["query_player", "local_game_rank"])
                    .groupby("query_player", group_keys=False)
                    .head(max(CONFIG.n_query_grid))
                    .copy()
                )
                player_df = (
                    limited.groupby(["query_player", "query_elo"], as_index=False)
                    .agg(
                        games_used=("game_index", "count"),
                        pred_elo_top1_mean=("pred_elo_top1", "mean"),
                        pred_elo_raw_mean=("pred_elo_raw", "mean"),
                        top1_unique=("pred_player_top1", "nunique"),
                    )
                    .copy()
                )
                cal_df = player_df.loc[player_df["query_player"].isin(cal_players)].copy()
                test_df = player_df.loc[player_df["query_player"].isin(test_players)].copy()
                slope, intercept = fit_linear_correction(cal_df, pred_col="pred_elo_raw_mean")
                test_df["pred_elo_corrected"] = slope * test_df["pred_elo_raw_mean"] + intercept
                test_df["seed"] = int(seed)
                test_df["memory_size"] = int(memory_size)
                test_df["regression_k"] = int(regression_k)
                test_df["linear_slope"] = slope
                test_df["linear_intercept"] = intercept
                player_rows.append(test_df)

                valid = test_df.dropna(subset=["pred_elo_raw_mean", "query_elo"]).copy()
                if valid.empty:
                    continue
                pred_range = float(valid["pred_elo_raw_mean"].max() - valid["pred_elo_raw_mean"].min())
                true_range = float(valid["query_elo"].max() - valid["query_elo"].min())
                metric_rows.append(
                    {
                        "seed": int(seed),
                        "memory_size": int(memory_size),
                        "regression_k": int(regression_k),
                        "players": int(len(valid)),
                        "mae_raw": float(np.mean(np.abs(valid["pred_elo_raw_mean"] - valid["query_elo"]))),
                        "mae_corrected": float(np.mean(np.abs(valid["pred_elo_corrected"] - valid["query_elo"]))),
                        "corr_raw": float(np.corrcoef(valid["query_elo"], valid["pred_elo_raw_mean"])[0, 1]) if valid["pred_elo_raw_mean"].nunique() > 1 else np.nan,
                        "pred_range": pred_range,
                        "true_range": true_range,
                        "range_ratio_pred_over_true": pred_range / true_range if true_range > 0 else np.nan,
                        "linear_slope": slope,
                        "linear_intercept": intercept,
                    }
                )

    metrics_out = pd.DataFrame(metric_rows)
    players_out = pd.concat(player_rows, ignore_index=True) if player_rows else pd.DataFrame()
    metrics_out.to_csv(RESCUE_MEMORY_SIZE_METRICS_CSV_PATH, index=False)
    players_out.to_csv(RESCUE_MEMORY_SIZE_PLAYERS_CSV_PATH, index=False)
    return metrics_out, players_out


if RUN_RESCUE_MEMORY_SIZE_EXPERIMENT:
    rescue_memory_metrics_df, rescue_memory_players_df = evaluate_rescue_memory_sizes()
    display(rescue_memory_metrics_df)

    summary = (
        rescue_memory_metrics_df.groupby(["memory_size", "regression_k"], as_index=False)
        .agg(
            mae_raw_mean=("mae_raw", "mean"),
            mae_corrected_mean=("mae_corrected", "mean"),
            corr_raw_mean=("corr_raw", "mean"),
            range_ratio_mean=("range_ratio_pred_over_true", "mean"),
        )
        .sort_values(["mae_raw_mean", "mae_corrected_mean"])
    )
    display(summary)

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.lineplot(data=summary, x="memory_size", y="mae_raw_mean", hue="regression_k", marker="o", ax=axes[0])
    axes[0].set_title("Rescue · MAE raw vs tamaño del banco")
    axes[0].set_xlabel("Jugadores base en memory bank")
    axes[0].set_ylabel("MAE raw")
    axes[0].grid(True, alpha=0.3)

    sns.lineplot(data=summary, x="memory_size", y="range_ratio_mean", hue="regression_k", marker="o", ax=axes[1])
    axes[1].set_title("Rescue · rango predicho / rango real")
    axes[1].set_xlabel("Jugadores base en memory bank")
    axes[1].set_ylabel("ratio de rango")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Experimento opcional no ejecutado. Cambia RUN_RESCUE_MEMORY_SIZE_EXPERIMENT=True para probar bancos N=10/25/50/100.")
    print(f"Si se ejecuta, guardará métricas en {RESCUE_MEMORY_SIZE_METRICS_CSV_PATH}")


## Artefactos generados

Los CSV quedan en `labs/notebooks/output/events/experimental_setup_122/`:

- `candidate_catalog.csv`: catálogo de candidatos con ELO.
- `base_pools_by_seed.csv`: pools base de `N=10` por seed.
- `query_pools_by_seed.csv`: jugadores-query por seed.
- `a2_seed_metrics.csv`: métricas por seed, split y `nq`.
- `a3_nq_metrics.csv`: resumen MAE vs `nq`.
- `seed_predictions/seed_<seed>_game_predictions.csv`: predicciones a nivel partida.
- `scatter_data_test.csv`: datos correctos del scatter test `nq=20`, reconstruidos con el split real.
- `player_nq_summary_all_splits.csv`: resumen por jugador, seed, split y `nq`.
- `prediction_collapse_diagnostics.csv`: rango y desviación de predicciones raw por seed.
- `random_individual_players_by_seed.csv`: 5 jugadores test aleatorios por seed para inspección individual.
- `diagnostics/`: curvas individuales y uso de jugadores ancla top-1 por seed.
